# Mini Project 1 — Analysis Notebook (MP1b)

**Name:** _Zhian Hu_  
**Dataset:** [ChatGPT reviews (daily updated)](https://www.kaggle.com/datasets/ashishkumarak/chatgpt-reviews-daily-updated) — `ashishkumarak/chatgpt-reviews-daily-updated`.  
**Date:** May 2026  

This notebook extends the Week 5 / A5 pandas exploration with **Section — Analysis**: Plotly visuals tied to MP1 analytical questions, exported as static `.png` files for Canvas / GitHub.

In [1]:
# Setup — run first (Plotly static export requires kaleido)
import subprocess
import sys


def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])


for pkg in ["pandas", "plotly", "kaleido", "kagglehub", "nbformat"]:
    try:
        __import__(pkg if pkg != "kagglehub" else "kagglehub")
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print("Setup complete.")

Setup complete.


/Users/hza25/Desktop/hcde530/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/hza25/Desktop/hcde530/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Section 1 — Overview

**Why this dataset:** Public app-store reviews bundle **stated sentiment (stars)**, **text**, and lightweight **social engagement (thumbs-up)** at scale — relevant for HCI / product sensing around generative AI tools.

**Three analytical questions (from MP1a / exploratory plan):**

1. **Polarity skew:** How are star ratings distributed — is sentiment dominated by positive reviews?
2. **Engagement vs. rating:** Do lower-star reviews draw more thumbs-ups on average (amplified complaints), or does engagement track positivity?
3. **Data completeness:** Which columns are unreliable because of systematic missing values (especially version metadata tied to rollout studies)?

**What a practitioner would do with findings:** Product and trust teams can prioritize complaint themes sampling with awareness of imbalance, temper conclusions that depend on exact app/version strings, and interpret community engagement independently from star averages.

---
## Section 2 — Data Profile

**Load reproducibly:** Matches A5 behavior — prefers `kagglehub.dataset_download`, then finds the largest non-demo CSV under that path.

**Profile questions (answered in prose after running the cells):**
- Rows/columns scale (expect ~1M reviews, eight core columns).
- `score`: 1–5 star ratings; `thumbsUpCount`: non‑negative integer engagement; timestamps in `at`; version fields partly missing.
- Structured missingness overlapping `reviewCreatedVersion` and `appVersion`.
- Focus columns: **`score`, `thumbsUpCount`** for Sections 3 charts 1–2; completeness for chart 3.

In [2]:
from pathlib import Path

import pandas as pd

_EXCLUDE = {"app_info.csv", "app_reviews_demo.csv", "reviews_mini.csv"}


def load_chatgpt_reviews_csv(data_dir: Path) -> tuple[pd.DataFrame, Path]:
    preferred = data_dir / "chatgpt_reviews_kaggle.csv"
    if preferred.exists():
        csv_path = preferred
    else:
        candidates = [
            p
            for p in data_dir.rglob("*.csv")
            if p.name not in _EXCLUDE and not p.name.startswith(".")
        ]
        if not candidates:
            raise FileNotFoundError(
                f"No CSV under {data_dir}. Run `kagglehub.dataset_download(...)`, "
                "or place chatgpt_reviews_kaggle.csv in this notebook folder."
            )
        csv_path = max(candidates, key=lambda p: p.stat().st_size)
    return pd.read_csv(csv_path), csv_path


_root = globals().get("path")

if _root is None:
    try:
        import kagglehub

        path = kagglehub.dataset_download(
            "ashishkumarak/chatgpt-reviews-daily-updated"
        )
    except Exception:
        path = Path(".")

dataset_root = Path(path)

df, _csv_used = load_chatgpt_reviews_csv(dataset_root)
rating_col = "score"

print("Loaded:", _csv_used.resolve())
print("Shape:", df.shape)
df.head(3)

Loaded: /Users/hza25/.cache/kagglehub/datasets/ashishkumarak/chatgpt-reviews-daily-updated/versions/644/chatgpt_reviews.csv
Shape: (1031275, 8)


,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion
0,6f3b0c1f-b36e-4deb-9b3e-35eddb4e81d8,Shan Janii,open the app ok,3,0,1.2026.125,2026-05-12 09:43:03,1.2026.125
1,90e71bdc-b914-43d9-822e-7f966eba2ea0,Mahesa Pritanika P,Very disappointed with the current model. Mode...,1,0,1.2026.125,2026-05-12 09:42:57,1.2026.125
2,2d4aab6d-e524-4d28-b734-656eb865047a,Shreeram Tarole,very very good work,5,0,1.2026.125,2026-05-12 09:42:51,1.2026.125


In [3]:
df.info()
missing_preview = df.isnull().sum()
missing_preview[missing_preview > 0]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031275 entries, 0 to 1031274
Data columns (total 8 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   reviewId              1031275 non-null  object
 1   userName              1031273 non-null  object
 2   content               1031259 non-null  object
 3   score                 1031275 non-null  int64 
 4   thumbsUpCount         1031275 non-null  int64 
 5   reviewCreatedVersion  956009 non-null   object
 6   at                    1031275 non-null  object
 7   appVersion            956009 non-null   object
dtypes: int64(2), object(6)
memory usage: 62.9+ MB


userName                    2
content                    16
reviewCreatedVersion    75266
appVersion              75266
dtype: int64

---
## Section 3 — Analysis (charts)

Each chart answers **one analytical question**. Figures are exported as PNGs next to this notebook (`A6/`).

Chart types intentionally differ (**donut / line+markers / treemap**) so each view matches both the metric and how we expect readers to compare values (parts-of-whole, ordered trend, relative gaps in quality).

### Chart 1 — Distribution of star ratings

**Question:** How concentrated are ratings at each star level?

**Chart type:** **Donut (`go.Pie` with hole)** — we care about **share of whole** (five slices that sum to 100%); wedge angle encodes proportion. That answers “skew” more directly than stacking bars once levels are exhaustive.

In [4]:
# Save PNG beside this notebook — run with working directory `A6` (recommended).
_OUT = Path.cwd().resolve()

counts = df[rating_col].value_counts().sort_index()
pie_labels = [f"{star}★" for star in counts.index]

fig1 = go.Figure(
    data=[
        go.Pie(
            labels=pie_labels,
            values=counts.values,
            hole=0.45,
            textinfo="label+percent",
            textposition="outside",
            sort=False,
        )
    ]
)
fig1.update_layout(
    title={
        "text": "Composition of ChatGPT Play Store ratings (share by star level)<br><sub>Wedge proportion = fraction of reviews; labels show percentages.</sub>",
        "x": 0.5,
        "xanchor": "center",
    },
    showlegend=False,
    margin=dict(t=130, l=40, r=40, b=80),
)

fig1.write_image(_OUT / "chart1_rating_distribution.png")
fig1.show()

**Finding:** Massive imbalance toward 5★ reviews → any subgroup comparison or sampling for themes must normalize or stratify consciously.

### Chart 2 — Mean thumbs-ups by rating

**Question:** Does thumb-based engagement correlate with polarity?

**Chart type:** **Line + markers (`go.Scatter`, `mode="lines+markers"`)** — star ratings form an **ordinal** scale; linking means with a line foregrounds whether engagement **systematically rises or falls** as stars increase (or stays flat—also a substantive pattern).

In [5]:
_OUT = Path.cwd().resolve()

means = (
    df.groupby(rating_col, as_index=False)["thumbsUpCount"]
    .mean()
    .sort_values(rating_col)
)

fig2 = go.Figure(
    data=[
        go.Scatter(
            x=means[rating_col],
            y=means["thumbsUpCount"],
            mode="lines+markers",
            line=dict(width=3, shape="linear"),
            marker=dict(size=12),
        )
    ]
)
fig2.update_layout(
    title="Average reader thumbs-ups by star rating (trend across ordered scale)",
    xaxis_title="Star rating (1–5 ordinal scale)",
    yaxis_title="Mean thumbs-up count (per review)",
    xaxis=dict(tickmode="linear", dtick=1),
)

fig2.write_image(_OUT / "chart2_mean_thumbs_by_rating.png")
fig2.show()

**Finding:** Means sit in a narrow band across stars (**relatively flat**). Interpretation: in this corpus, thumbs-ups are **not** a loud signal separating complaints from praise — qualitative reading still matters.

**Null / dull result clause:** Flat averages are informative: they constrain what engagement alone can diagnose.

### Chart 3 — Missing values by column

**Question:** Which fields are systematically incomplete — especially version metadata affecting temporal or rollout slicing?

**Chart type:** **Treemap (`plotly.express.treemap`)** — rectangles **tile** by missing-cell count so we compare **severities** hierarchically (`column → missing_cells`). Fits a **single-level category set** whose names are clearer as tiles than repeating bar ranks.

In [6]:
_OUT = Path.cwd().resolve()

missing_series = df.isnull().sum()
missing_series = missing_series[missing_series > 0].sort_values(ascending=False)

missing_tbl = pd.DataFrame(
    {"column": missing_series.index.astype(str), "missing_cells": missing_series.values}
)

fig3 = px.treemap(
    missing_tbl,
    path=["column"],
    values="missing_cells",
    title="Missing cells by column (tile area proportional to missing count)",
)
fig3.update_traces(textinfo="label+value")
fig3.update_layout(margin=dict(t=48, l=12, r=12, b=12))

fig3.write_image(_OUT / "chart3_missing_values_by_column.png")
fig3.show()

**Finding:** **`reviewCreatedVersion`** and **`appVersion`** miss the same counts (aligned missingness pattern) vs. near-complete **`score`** / timestamps — tie version-aware analyses with explicit exclusions or “unknown bucket” labeling.